In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

In [ ]:
import json
RAW_JOBS_FILE = PROJECT_ROOT/"data"/"raw"/"skillab_jobs_isco25.json"

with open('skillab_jobs_isco25.json', 'r', encoding='utf-8') as f:
    jobs = json.load(f)

print(f"Φορτώθηκαν επιτυχώς {len(jobs)} αγγελίες!")

Φορτώθηκαν επιτυχώς 816 αγγελίες!


In [ ]:
first_job = jobs[0]

job_title = first_job.get('title')
print(f"Αγγελία: {job_title}\n")

esco_skills_links = first_job.get('skills', [])

print(f"Βρέθηκαν {len(esco_skills_links)} επίσημα ESCO Skills. Οι κωδικοί τους είναι:")

for link in esco_skills_links:
    clean_skill_id = link.split('/')[-1]
    print(f"-> {clean_skill_id}")

Αγγελία: Software Engineer (.NET & Angular)

Βρέθηκαν 8 επίσημα ESCO Skills. Οι κωδικοί τους είναι:
-> 1d86f05e-e9cc-40ce-99d8-2b21cc71b16b
-> 1ffac4ac-fda7-407d-8ed1-ca8f4a8dc146
-> 5b26f08b-88bc-45f0-b901-530d7786466b
-> 89f6560b-2194-45c9-9ece-d33049a73eef
-> a8797f21-1af0-4c5e-98ca-bf1442929363
-> b1ef509e-f191-405a-8419-51663b08922e
-> d9e5349e-8791-49c2-8ba4-839fdd1606c2
-> ea4ebfa1-e17a-4416-ac54-955f33e6ade7


In [ ]:
clean_jobs_for_neo4j = []

for job in jobs:
    job_id = job.get('id')
    title = job.get('title')
    
    esco_links = job.get('skills', [])
    
    clean_skills = []

    if esco_links:
        for link in esco_links:
            skill_id = link.split('/')[-1] 
            clean_skills.append(skill_id)
            
    clean_job = {
        "job_id": job_id,
        "title": title,
        "skills": clean_skills
    }

    clean_jobs_for_neo4j.append(clean_job)

OUTPUT_FILE = PROJECT_ROOT/"data"/"processed"/"neo4j_ready_jobs.json"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(clean_jobs_for_neo4j, f, indent=4, ensure_ascii=False)

print(f"🎉 Επιτυχία! Καθαρίστηκαν {len(clean_jobs_for_neo4j)} αγγελίες.")
print("Το νέο αρχείο 'neo4j_ready_jobs.json' δημιουργήθηκε στον φάκελό σου!")

🎉 Επιτυχία! Καθαρίστηκαν 816 αγγελίες.
Το νέο αρχείο 'neo4j_ready_jobs.json' δημιουργήθηκε στον φάκελό σου!


In [ ]:
import json
import os
from dotenv import load_dotenv
from pathlib import Path
from neo4j import GraphDatabase

load_dotenv()

PROJECT_ROOT = Path.cwd().parent
DATA_FILE = PROJECT_ROOT/"data"/"processed"/"neo4j_ready_jobs.json"

URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USER")
PASSWORD = os.getenv("NEO4J_PASSWORD")

if not URI or not USER or not PASSWORD:
    raise RuntimeError("Neo4j environment variables are required.")

AUTH = (USER, PASSWORD)


with open(DATA_FILE, "r", encoding="utf-8") as f:
    jobs_data = json.load(f)

def create_graph_data(tx, job_data):
    query = """
    MERGE (j:JobPosting {job_id: $job_id})
    SET j.title = $title
    WITH j
    UNWIND $skills AS skill_id
    
    MERGE (s:Skill {esco_id: skill_id})
    MERGE (j)-[:REQUIRES]->(s)
    """
    tx.run(query, job_id=job_data['job_id'], title=job_data['title'], skills=job_data['skills'])

print("Ξεκινάει η δημιουργία του Γράφου. Αυτό μπορεί να πάρει μερικά δευτερόλεπτα...")

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    with driver.session() as session:
        for job in jobs_data:
            session.execute_write(create_graph_data, job)

print(f"✅ Ολοκληρώθηκε! Φορτώθηκαν επιτυχώς {len(jobs_data)} αγγελίες και όλα τα skills τους στη Neo4j.")

Ξεκινάει η δημιουργία του Γράφου. Αυτό μπορεί να πάρει μερικά δευτερόλεπτα...
✅ Ολοκληρώθηκε! Φορτώθηκαν επιτυχώς 816 αγγελίες και όλα τα skills τους στη Neo4j.


In [2]:
%pip install neo4j

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import csv
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USER")
PASSWORD = os.getenv("NEO4J_PASSWORD")

if not URI or not USER or not PASSWORD:
    raise RuntimeError("Neo4j environment variables are required.")

AUTH = (USER, PASSWORD)

skills_mapping = []
print("Διαβάζω το λεξικό του ESCO...")

with open('skills_en.csv', mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        uri = row.get('conceptUri', '')
        if uri:
            skill_id = uri.split('/')[-1]
            skill_name = row.get('preferredLabel', 'Άγνωστο Skill')
            skills_mapping.append({"skill_id": skill_id, "skill_name": skill_name})

print(f"Βρέθηκαν {len(skills_mapping)} δεξιότητες στο λεξικό. Ξεκινάει η ενημέρωση του Γράφου...")

def enrich_skills(tx, mapping_batch):
    query = """
    UNWIND $batch AS skill_data
    MATCH (s:Skill {esco_id: skill_data.skill_id})
    SET s.name = skill_data.skill_name
    """
    tx.run(query, batch=mapping_batch)

batch_size = 1000
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    with driver.session() as session:
        for i in range(0, len(skills_mapping), batch_size):
            batch = skills_mapping[i:i+batch_size]
            session.execute_write(enrich_skills, batch)

print("✅ Επιτυχία! Όλα τα skills στον Γράφο σου απέκτησαν πλέον το πραγματικό τους όνομα!")

Διαβάζω το λεξικό του ESCO...
Βρέθηκαν 13960 δεξιότητες στο λεξικό. Ξεκινάει η ενημέρωση του Γράφου...
✅ Επιτυχία! Όλα τα skills στον Γράφο σου απέκτησαν πλέον το πραγματικό τους όνομα!
